In [ ]:
!nvidia-smi

Sun May  4 18:41:25 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.07                 Driver Version: 566.07         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650      WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   40C    P8              5W /   65W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# imstall these required packagess once
# !pip install -q kaggle
# !pip install -q nltk
# !pip install -q tqdm
# !pip install tensorflow



# Imports
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

# only use when working using gpu of google
import kagglehub
from google.colab import files
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D,  Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix


print('imports done!')

imports done!


In [ ]:
# loading the dataset of facial expressions from kaggle !


path = kagglehub.dataset_download("msambare/fer2013")

print("Path to dataset files: ", path)
print("Contents of dataset directory:")
print(os.listdir(path))

# path = r"C:\Users\cai\Documents\6th Semester - Spring 2025\Deep Learning\Project\ferdataset"

# print("Path to dataset files:", path)
# print("Contents of dataset directory:")
# print(os.listdir(path))

# image paths for the dataset
train_dir = os.path.join(path, "train")
val_dir = os.path.join(path, "test")

Path to dataset files:  /kaggle/input/fer2013
Contents of dataset directory:
['test', 'train']


In [ ]:
# since ResNet expects images of size 224x224 RGB we have to resize & normalize!
# do for training and test dataset

# preprocessing data by resizing, normalizing and audmenting the data

image_size = (224, 224)
batch_size = 32

# do data augmentation to the training set for better generalization - this way it wont be dependent on exact inputs
# so we add trasnformations such as rotation, zoom, rescale etc.

train_datagenerator = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# training set
train_generator = train_datagenerator.flow_from_directory(
    train_dir,
    target_size=image_size,
    batch_size=batch_size,
    color_mode='rgb',
    class_mode='categorical'
)

# validation set
val_generator = train_datagenerator.flow_from_directory(
    val_dir,
    target_size=image_size,
    batch_size=batch_size,
    color_mode='rgb',
    class_mode='categorical',
    shuffle=False
)


Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


# **Parameter Training to find the Best Parameters to Train with !**
DO NOT RUN THIS!!! THIS IS DONEEEEEEE
Note: This ran for 5hrs

In [ ]:
# Grid search values
etas = [1e-5, 1e-4, 1e-3]
epochs_list = [5, 10, 15]

# Best tracking
best_val_acc = 0.0
best_model = None
best_eta = None
best_epochs = None

for eta in etas:
    for n_epochs in epochs_list:

        print(f"Training model with learning rate = {eta}, epochs = {n_epochs}")

        # Build model
        base_res_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        base_res_model.trainable = False

        x = base_res_model.output
        x = GlobalAveragePooling2D()(x)
        x = Dropout(0.5)(x)
        x = Dense(128, activation='relu')(x)
        x = Dropout(0.3)(x)
        predictions = Dense(train_generator.num_classes, activation='softmax')(x)

        model = Model(inputs=base_res_model.input, outputs=predictions)

        # Compile model
        model.compile(
            optimizer=Adam(learning_rate=eta),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        # Train
        early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True, verbose=0)
        history = model.fit(
            train_generator,
            validation_data=val_generator,
            epochs=n_epochs,
            callbacks=[early_stop],
            verbose=1
        )

        # Check performance
        val_acc = max(history.history['val_accuracy'])
        print(f"Validation Accuracy: {val_acc:.4f}")

        # Save best
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = model
            best_eta = eta
            best_epochs = n_epochs
            print("New best model found!")


# make sure to save best model
best_model.save("best_facial_expression_model.keras")
print("\nBest Model Summary:")
print(f"Best Learning Rate: {best_eta}")
print(f"Best Number of Epochs: {best_epochs}")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print("Model saved as 'best_facial_expression_model.keras'")

# Download from google colab to my laptop - i do this bcs it doesnt show up sometimes in google collab's temp contents folder
# from google.colab import files
# files.download("best_facial_expression_model.keras")

model.save("best_facial_expression_model.keras")
print("Saved to current directory")


Training model with learning rate = 1e-05, epochs = 5
Epoch 1/5
898/898 ━━━━━━━━━━━━━━━━━━━━ 526s 556ms/step - accuracy: 0.1426 - loss: 2.5028 - val_accuracy: 0.2471 - val_loss: 1.8472
Epoch 2/5
898/898 ━━━━━━━━━━━━━━━━━━━━ 489s 544ms/step - accuracy: 0.1901 - loss: 2.0602 - val_accuracy: 0.2470 - val_loss: 1.8486
Epoch 3/5
898/898 ━━━━━━━━━━━━━━━━━━━━ 491s 547ms/step - accuracy: 0.1964 - loss: 1.9800 - val_accuracy: 0.2473 - val_loss: 1.8381
Epoch 4/5
898/898 ━━━━━━━━━━━━━━━━━━━━ 490s 545ms/step - accuracy: 0.1995 - loss: 1.9402 - val_accuracy: 0.2473 - val_loss: 1.8288
Epoch 5/5
898/898 ━━━━━━━━━━━━━━━━━━━━ 482s 536ms/step - accuracy: 0.1976 - loss: 1.9106 - val_accuracy: 0.2471 - val_loss: 1.8325
Validation Accuracy: 0.2473
New best model found!
Training model with learning rate = 1e-05, epochs = 10
Epoch 1/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 549s 600ms/step - accuracy: 0.1894 - loss: 2.3924 - val_accuracy: 0.1335 - val_loss: 1.8765
Epoch 2/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 493s 549ms/ste

In [ ]:
# Grid search values continued
etas = [1e-4, 1e-3]
epochs_list = [5, 10, 15]

# Best tracking
best_val_acc = 0.0
best_model = None
best_eta = None
best_epochs = None

for eta in etas:
    for n_epochs in epochs_list:

        if eta == 1e-4 and (n_epochs == 5 or n_epochs == 10):
          continue

        print(f"Training model with learning rate = {eta}, epochs = {n_epochs}")

        # Build model
        base_res_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        base_res_model.trainable = False

        x = base_res_model.output
        x = GlobalAveragePooling2D()(x)
        x = Dropout(0.5)(x)
        x = Dense(128, activation='relu')(x)
        x = Dropout(0.3)(x)
        predictions = Dense(train_generator.num_classes, activation='softmax')(x)

        model = Model(inputs=base_res_model.input, outputs=predictions)

        # Compile model
        model.compile(
            optimizer=Adam(learning_rate=eta),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        # Train
        early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True, verbose=0)
        history = model.fit(
            train_generator,
            validation_data=val_generator,
            epochs=n_epochs,
            callbacks=[early_stop],
            verbose=1
        )

        # Check performance
        val_acc = max(history.history['val_accuracy'])
        print(f"Validation Accuracy: {val_acc:.4f}")

        # Save best
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model = model
            best_eta = eta
            best_epochs = n_epochs
            print("New best model found!")


# make sure to save best model
best_model.save("best_facial_expression_model.keras")
print("\nBest Model Summary:")
print(f"Best Learning Rate: {best_eta}")
print(f"Best Number of Epochs: {best_epochs}")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print("Model saved as 'best_facial_expression_model.keras'")

# Download from google colab to my laptop - i do this bcs it doesnt show up sometimes in google collab's temp contents folder
# from google.colab import files
# files.download("best_facial_expression_model.keras")

model.save("best_facial_expression_model.keras")
print("Saved to current directory")

Training model with learning rate = 0.001, epochs = 5
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
  3/898 ━━━━━━━━━━━━━━━━━━━━ 1:20:41 5s/step - accuracy: 0.0764 - loss: 2.7362

KeyboardInterrupt: 

In [ ]:
# Reload and evaluate
saved_model = tf.keras.models.load_model("best_facial_expression_model.keras")
score = saved_model.evaluate(val_generator)

print("Testing Loss:", score[0], "Testing Accuracy:", score[1])

# **Plug in the Best Parameters found and Train them for longer !**

In [ ]:
# ResNet Transfer Layer Model! (starting Transfer Learning)
# we chose this because its already pretrained with imagenet - only planning to train the last layer
# faster and more efficient for our project

# well use Adam Optimizer in ResNet 50

# loading ResNet50 w/o last layer then we freeze!
base_res_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_res_model.trainable = False

# should we do avgpooling or maxpooling???? idk yettt

x = base_res_model.output
x = GlobalAveragePooling2D()(x)
# x = MaxPooling2D(pool_size=(7, 7))(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)

predictions = Dense(train_generator.num_classes, activation='softmax')(x)

# build & save the model !!
model = Model(inputs=base_res_model.input, outputs=predictions)

# Compile it
model.compile(optimizer=Adam(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Summary
model.summary()



In [ ]:
# doing early stopping so that it stops training when the valdation loss stops improving
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# train the model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stopping]
)

# Save the model to a file
model.save('facial_exp_resnet50_model.keras')  #Will create a directory that has all the related files and saves everything there

# files.download("facial_exp_resnet50_model.keras")
saved_model = tf.keras.models.load_model('facial_exp_resnet50_model.keras')
score = saved_model.evaluate(val_generator)

print("Testing Loss:", score[0], "Testing Accuracy:", score[1])
print("Model saved as 'facial_expression_resnet_model'")

Epoch 1/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 471s 513ms/step - accuracy: 0.1630 - loss: 2.3791 - val_accuracy: 0.1780 - val_loss: 1.8295
Epoch 2/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 450s 501ms/step - accuracy: 0.1868 - loss: 2.0995 - val_accuracy: 0.1782 - val_loss: 1.8310
Epoch 3/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 447s 498ms/step - accuracy: 0.1864 - loss: 2.0231 - val_accuracy: 0.1970 - val_loss: 1.8326
Epoch 4/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 442s 492ms/step - accuracy: 0.1931 - loss: 1.9638 - val_accuracy: 0.2473 - val_loss: 1.8355


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

225/225 ━━━━━━━━━━━━━━━━━━━━ 96s 397ms/step - accuracy: 0.1798 - loss: 1.8239
Testing Loss: 1.828909993171692 Testing Accuracy: 0.17887991666793823
Model saved as 'facial_expression_resnet_model'


In [ ]:
loss, accuracy = model.evaluate(val_generator)
print(f"✅ Test Loss: {loss:.4f}")
print(f"✅ Test Accuracy: {accuracy:.4f}")

225/225 ━━━━━━━━━━━━━━━━━━━━ 96s 427ms/step - accuracy: 0.1829 - loss: 1.8199
✅ Test Loss: 1.8291
✅ Test Accuracy: 0.1797


In [ ]:
# converts softmax probabilities to class labels!

# Predict probabilities
y_pred_probs = model.predict(val_generator)
y_pred = np.argmax(y_pred_probs, axis=1)

# True labels
y_true = val_generator.classes

# Class labels
class_labels = list(val_generator.class_indices.keys())

# Classification report
print("Classification Report:\n", classification_report(y_true, y_pred, target_names=class_labels))

225/225 ━━━━━━━━━━━━━━━━━━━━ 95s 403ms/step
Classification Report:
               precision    recall  f1-score   support

       angry       0.00      0.00      0.00       958
     disgust       0.00      0.00      0.00       111
        fear       0.00      0.00      0.00      1024
       happy       0.25      0.14      0.18      1774
     neutral       0.17      0.86      0.29      1233
         sad       0.00      0.00      0.00      1247
    surprise       0.00      0.00      0.00       831

    accuracy                           0.18      7178
   macro avg       0.06      0.14      0.07      7178
weighted avg       0.09      0.18      0.09      7178



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from tensorflow.keras.preprocessing import image

img_path = "/content/man.jpg"
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)
pred_class = class_labels[np.argmax(pred)]

print(f"Predicted Expression: {pred_class}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
Predicted Expression: neutral
